<a href="https://colab.research.google.com/github/bintanghan07/data-science-2026/blob/main/Pertemuan12_Bintang%20Hanifatul%20Manunggal_250401020095.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PRAKTIKUM PERTEMUAN 12 DATA SCIENCE**

Nama : Bintang Hanifatul Manunggal

Nim :  250401020095

Kelas : IF405

### **Langkah 1: Generate & Eksplorasi Dataset Transaksi**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Daftar produk
np.random.seed(42)

produk = [
    "Roti", "Selai", "Susu", "Sereal", "Telur",
    "Keju", "Kopi", "Gula", "Teh", "Mentega"
]

# Membuat 50 transaksi
transaksi = []

for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(
        list(np.random.choice(produk, n_item, replace=False))
    )

# Menambahkan pola: Roti sering dibeli bersama Selai
for i in range(20):
    if "Roti" in transaksi[i] and "Selai" not in transaksi[i]:
        transaksi[i].append("Selai")

# Menampilkan informasi dataset
print("Contoh transaksi:")
for i, t in enumerate(transaksi[:3], start=1):
    print(f"Transaksi {i}: {t}")

print("\nJumlah transaksi:", len(transaksi))

Contoh transaksi:
Transaksi 1: [np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai']
Transaksi 2: [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')]
Transaksi 3: [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]

Jumlah transaksi: 50


Dataset transaksi sintetis berhasil dibuat dengan 50 transaksi yang masing-masing berisi 2–5 produk. Selain itu, ditambahkan pola bahwa produk Roti sering dibeli bersama Selai, sehingga terdapat hubungan antarproduk yang dapat ditemukan pada proses Association Rule Mining

### **Langkah 2: One-Hot Encoding Transaksi**

In [2]:
from mlxtend.preprocessing import TransactionEncoder
import pandas as pd

# Melakukan one-hot encoding pada data transaksi
te = TransactionEncoder()

te_ary = te.fit(transaksi).transform(transaksi)

# Membuat DataFrame hasil encoding
df = pd.DataFrame(
    te_ary,
    columns=te.columns_
)

# Menampilkan 5 data pertama
print("Hasil One-Hot Encoding:")
print(df.head())

Hasil One-Hot Encoding:
    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


di tahap ini, data transaksi diubah menjadi bentuk one-hot encoding. Setiap kolom merepresentasikan satu jenis produk, sedangkan setiap baris menunjukkan satu transaksi. Nilai True menandakan bahwa produk tersebut dibeli pada transaksi tersebut, sedangkan False menunjukkan produk tidak dibeli. Hasil transformasi ini menghasilkan data dalam format biner

### **Langkah 3: Cari Frequent Itemset dengan Apriori**

In [11]:
import warnings
warnings.filterwarnings("ignore")

In [10]:

import os

os.environ["PYTHONWARNINGS"] = "ignore"

import warnings

warnings.simplefilter("ignore")
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    module="jupyter_client"
)


In [12]:
from mlxtend.frequent_patterns import apriori

# Mencoba beberapa nilai minimum support
for ms in [0.05, 0.10, 0.20]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f"min_support = {ms}: {len(freq)} itemset ditemukan")

# Menggunakan nilai minimum support yang dianggap paling sesuai
freq_items = apriori(df, min_support=0.10, use_colnames=True)

# Mengurutkan berdasarkan nilai support tertinggi
freq_items = freq_items.sort_values("support", ascending=False)

# Menampilkan 10 itemset teratas
print("\n10 Frequent Itemset Teratas:")
print(freq_items.head(10))

min_support = 0.05: 74 itemset ditemukan
min_support = 0.1: 44 itemset ditemukan
min_support = 0.2: 13 itemset ditemukan

10 Frequent Itemset Teratas:
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Selai, Teh)


Berdasarkan hasil diatas, jumlah frequent itemset dipengaruhi oleh nilai min_support yang digunakan. Pada min_support = 0,05 ditemukan 74 itemset, pada 0,10 ditemukan 44 itemset, dan pada 0,20 hanya ditemukan 13 itemset. Semakin besar nilai min_support, semakin sedikit itemset yang memenuhi syarat sebagai frequent itemset.

Pada praktikum ini dipilih min_support = 0,10 karena menghasilkan jumlah itemset yang cukup untuk dianalisis. Hasil menunjukkan bahwa Selai merupakan produk yang paling sering muncul dengan nilai support 0,52, diikuti oleh Teh (0,46) dan Mentega (0,42). Selain itu, kombinasi Selai dan Teh memiliki support 0,24, sehingga pasangan produk tersebut cukup sering muncul dalam transaksi.

### **Langkah 4: Bentuk & Saring Aturan Asosiasi**

In [13]:
from mlxtend.frequent_patterns import association_rules

# Membentuk association rules berdasarkan confidence
rules = association_rules(
    freq_items,
    metric="confidence",
    min_threshold=0.5
)

# Menyaring aturan dengan nilai lift > 1
rules = rules[rules["lift"] > 1]

# Mengurutkan berdasarkan nilai lift tertinggi
rules = rules.sort_values(
    by="lift",
    ascending=False
)

# Menampilkan 10 aturan asosiasi teratas
print("10 Association Rules Teratas:")
print(
    rules[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].head(10)
)

10 Association Rules Teratas:
         antecedents consequents  support  confidence      lift
8        (Teh, Keju)     (Telur)     0.12    0.857143  2.380952
14  (Mentega, Selai)      (Kopi)     0.10    0.625000  1.953125
12      (Roti, Gula)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
9       (Teh, Telur)      (Keju)     0.12    0.600000  1.764706
15     (Selai, Kopi)   (Mentega)     0.10    0.714286  1.700680
10     (Keju, Telur)       (Teh)     0.12    0.750000  1.630435
11     (Selai, Gula)      (Roti)     0.10    0.500000  1.562500
13   (Mentega, Kopi)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


Berdasarkan hasil diatas, aturan dengan nilai lift tertinggi adalah (Teh, Keju) → (Telur) dengan nilai lift sebesar 2,381 dan confidence sebesar 0,857. Hal ini menunjukkan bahwa pelanggan yang membeli Teh dan Keju memiliki peluang yang cukup tinggi untuk juga membeli Telur.

Selain itu, aturan Roti → Selai juga ditemukan dengan confidence sebesar 0,688 dan lift sebesar 1,322. Hasil ini sesuai dengan pola yang sengaja ditambahkan pada dataset, yaitu Roti sering dibeli bersama Selai. Aturan-aturan tersebut dapat dimanfaatkan sebagai dasar strategi promosi, seperti penempatan produk berdekatan atau pembuatan paket penjualan agar dapat meningkatkan nilai transaksi pelanggan

### **Langkah 5: Rekomender Sederhana dengan Content-Based Filtering**

In [14]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Membuat katalog produk
katalog = pd.DataFrame({
    "produk": produk,
    "kategori": [
        "Bakery", "Bakery", "Dairy", "Bakery", "Dairy",
        "Dairy", "Minuman", "Bumbu", "Minuman", "Dairy"
    ]
})

# One-Hot Encoding kategori
fitur = pd.get_dummies(katalog["kategori"])

# Menghitung cosine similarity
sim_matrix = cosine_similarity(fitur)

# Fungsi rekomendasi produk serupa
def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog["produk"] == nama_produk][0]

    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)

    # Menghapus produk yang sama dari hasil rekomendasi
    skor = [s for s in skor if s[0] != idx][:top_n]

    return katalog.iloc[[i for i, _ in skor]]["produk"].tolist()

# Contoh rekomendasi
print("Mirip dengan Roti:", rekomendasi_serupa("Roti"))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


Berdasarkan hasil diatas, produk yang direkomendasikan untuk Roti adalah Selai, Sereal, dan Susu. Rekomendasi diberikan berdasarkan kemiripan karakteristik produk menggunakan cosine similarity pada fitur kategori yang telah diubah menjadi bentuk one-hot encoding.

Hasil ini menunjukkan bahwa sistem mampu memberikan rekomendasi produk yang memiliki karakteristik serupa dengan produk yang dipilih. Rekomendasi seperti ini dapat dimanfaatkan untuk membantu pelanggan menemukan produk terkait serta meningkatkan peluang penjualan melalui fitur rekomendasi produk.

### Langkah 6: Bandingkan Kedua Pendekatan

In [15]:
# Produk yang akan dicari rekomendasinya
produk_target = "Roti"

# Mencari aturan asosiasi yang memiliki antecedent mengandung produk target
rules_terkait = rules[
    rules["antecedents"].apply(lambda x: produk_target in x)
]

# Menampilkan rekomendasi dari Association Rules
print("Rekomendasi dari Association Rules:")
print(
    rules_terkait[
        ["consequents", "lift"]
    ].head()
)

# Menampilkan rekomendasi dari Content-Based Filtering
print("\nRekomendasi dari Content-Based:")
print(rekomendasi_serupa(produk_target))

Rekomendasi dari Association Rules:
   consequents      lift
12     (Selai)  1.923077
1      (Selai)  1.322115

Rekomendasi dari Content-Based:
['Selai', 'Sereal', 'Susu']


Berdasarkan hasil yang diperoleh, Association Rules merekomendasikan produk Selai untuk pelanggan yang membeli Roti, dengan nilai lift tertinggi sebesar 1,923. Hal ini menunjukkan bahwa Roti dan Selai memiliki hubungan pembelian yang cukup kuat berdasarkan riwayat transaksi pelanggan.

Sementara itu, Content-Based Filtering merekomendasikan Selai, Sereal, dan Susu karena produk-produk tersebut memiliki karakteristik atau kategori yang mirip dengan Roti. Kedua pendekatan sama-sama merekomendasikan Selai, sehingga hasilnya dapat dikatakan cukup konsisten.

Association Rules lebih cocok digunakan untuk menemukan produk yang sering dibeli secara bersamaan, sedangkan Content-Based Filtering lebih sesuai untuk merekomendasikan produk yang memiliki karakteristik serupa. Menggabungkan kedua metode (hybrid) dapat menghasilkan rekomendasi yang lebih lengkap dan relevan bagi pelanggan

# **Kesimpulan**

Pada praktikum pertemuan 11 ini, proses Association Rule Mining dan Content-Based Filtering berhasil diterapkan pada dataset transaksi sintetis. Dataset terdiri dari 50 transaksi dengan beberapa pola pembelian yang sengaja dibuat, seperti Roti yang sering dibeli bersama Selai. Seluruh data transaksi kemudian diubah ke bentuk one-hot encoding agar dapat diproses menggunakan algoritma Apriori

Hasil Apriori menunjukkan bahwa nilai min_support memengaruhi jumlah frequent itemset yang ditemukan. Pada min_support 0,10 diperoleh 44 frequent itemset sehingga nilai tersebut dipilih karena menghasilkan jumlah itemset yang cukup untuk dianalisis. Selanjutnya, Association Rules menghasilkan beberapa aturan asosiasi, salah satunya Roti → Selai, yang menunjukkan bahwa kedua produk sering dibeli secara bersamaan

Pada pendekatan Content-Based Filtering, sistem memberikan rekomendasi produk berdasarkan kemiripan kategori menggunakan cosine similarity. Untuk produk Roti, sistem merekomendasikan Selai, Sereal, dan Susu. Jika dibandingkan, kedua pendekatan sama-sama merekomendasikan Selai sehingga hasilnya cukup konsisten. Secara keseluruhan, Association Rule Mining lebih cocok untuk menemukan pola pembelian berdasarkan riwayat transaksi, sedangkan Content-Based Filtering lebih sesuai untuk merekomendasikan produk yang memiliki karakteristik serupa. Kombinasi kedua metode dapat menghasilkan sistem rekomendasi yang lebih relevan dan bermanfaat bagi pelanggan
